# EA ONNX Model Training — Kaggle Notebook

End-to-end pipeline for training ONNX models compatible with the
**MultiStrategyAutonomousEA** MQL5 expert advisor.

| Spec | Value |
|------|-------|
| Input shape | `(1, 60, 57)` — (batch, seq_len, feat_dim) |
| Output shape | `(1, 3)` — (batch, N_CLASSES) raw logits |
| Classes | 0=SELL, 1=NEUTRAL, 2=BUY |
| Scaler | StandardScaler — 57 means + 57 scales saved as `scaler.bin` |
| ONNX opset | 12 |

---

## Cell 1 — Setup & Configuration

In [ ]:
# ============================================================
# Cell 1 — Setup & Configuration
# ============================================================

import os
import sys
import json
import struct
import warnings
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# --- Install required packages (Kaggle has most pre-installed) ---
import subprocess
for pkg in ["onnx", "onnxruntime", "optuna", "shap"]:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report,
)
from scipy.stats import spearmanr, norm

import onnx
import onnxruntime as ort

print(f"PyTorch  : {torch.__version__}")
print(f"ONNX     : {onnx.__version__}")
print(f"ORT      : {ort.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")


# --- Configuration ---
@dataclass
class Config:
    # Data
    data_path: str = "/kaggle/input/ea-training-data/training_data.csv"  # Kaggle dataset path
    use_precomputed_features: bool = True   # True if CSV has feature_00..feature_56 columns
    symbol_col: str = "symbol"
    date_col: str = "date"

    # Architecture
    seq_len: int = 60
    n_features: int = 57
    n_classes: int = 3

    # Triple-barrier labeling
    tb_k: float = 1.5          # ATR multiplier for profit/stop barriers
    tb_vertical_bars: int = 20  # Max holding period

    # CUSUM event filter
    cusum_threshold: float = 1.0

    # Train / val / test split
    train_ratio: float = 0.70
    val_ratio: float = 0.15
    # test_ratio is implicit: 1 - train - val

    # Training
    batch_size: int = 64
    epochs: int = 80
    lr: float = 3e-4
    weight_decay: float = 1e-4
    patience: int = 15          # Early stopping patience
    grad_clip: float = 1.0
    amp: bool = True            # Mixed precision

    # CPCV
    cpcv_n_splits: int = 6
    cpcv_n_test: int = 2
    cpcv_purge: int = 5
    cpcv_embargo: int = 5
    psr_threshold: float = 0.95

    # Deployment gate
    min_ic: float = 0.02

    # Output
    output_dir: str = "/kaggle/working/"
    onnx_opset: int = 12

    # Reproducibility
    seed: int = 42

cfg = Config()

# Override data_path for local runs
if not Path(cfg.data_path).exists():
    local_candidates = list(Path(".").rglob("training_data.csv"))
    if local_candidates:
        cfg.data_path = str(local_candidates[0])
        print(f"Using local data: {cfg.data_path}")
    else:
        print("WARNING: No training CSV found. Set cfg.data_path manually.")

def set_seed(seed: int = 42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(cfg.seed)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nDevice: {DEVICE}")
print(f"Config: seq_len={cfg.seq_len}, n_features={cfg.n_features}, n_classes={cfg.n_classes}")

## Cell 2 — Data Loading

In [ ]:
# ============================================================
# Cell 2 — Data Loading
# ============================================================

df = pd.read_csv(cfg.data_path)
df[cfg.date_col] = pd.to_datetime(df[cfg.date_col])

# Detect feature columns
feature_cols = sorted(
    [c for c in df.columns if c.startswith("feature_")],
    key=lambda x: int(x.split("_")[1]),
)
cfg.use_precomputed_features = len(feature_cols) == cfg.n_features

print(f"Shape          : {df.shape}")
print(f"Date range     : {df[cfg.date_col].min()} → {df[cfg.date_col].max()}")
print(f"Symbols        : {df[cfg.symbol_col].nunique() if cfg.symbol_col in df.columns else 'N/A'}")
if cfg.symbol_col in df.columns:
    print(f"Symbol counts  :\n{df[cfg.symbol_col].value_counts().to_string()}")
print(f"Feature cols   : {len(feature_cols)} ({'precomputed' if cfg.use_precomputed_features else 'will compute'})")
print(f"Columns        : {list(df.columns[:10])}{'...' if len(df.columns) > 10 else ''}")
df.head()

## Cell 3 — Feature Engineering (Python-side)

Replicates the 57 features from `AIFeatureVectorBuilder.mqh` and `data_pipeline.py`.
If the CSV already contains `feature_00..feature_56` columns (exported by
`TrainingDataExporter.mq5`), this step is skipped.

In [ ]:
# ============================================================
# Cell 3 — Feature Engineering
# ============================================================

def _ema(x: np.ndarray, period: int) -> np.ndarray:
    alpha = 2.0 / (period + 1)
    result = np.empty_like(x, dtype=np.float64)
    result[0] = x[0]
    for i in range(1, len(x)):
        result[i] = alpha * x[i] + (1 - alpha) * result[i - 1]
    return result


def _sma(x: np.ndarray, period: int) -> np.ndarray:
    return pd.Series(x).rolling(period, min_periods=1).mean().values


def _rsi(close: np.ndarray, period: int) -> np.ndarray:
    delta = np.diff(close, prepend=close[0]).astype(np.float64)
    avg_gain = _ema(np.maximum(delta, 0.0), period)
    avg_loss = _ema(np.maximum(-delta, 0.0), period)
    rs = avg_gain / (avg_loss + 1e-9)
    return (100.0 - 100.0 / (1.0 + rs)) / 100.0


def _bb_pct_b(close: np.ndarray, period: int = 20, mult: float = 2.0) -> np.ndarray:
    mid = _sma(close, period)
    std = pd.Series(close).rolling(period, min_periods=1).std(ddof=0).fillna(0).values
    upper = mid + mult * std
    lower = mid - mult * std
    return (close - lower) / (upper - lower + 1e-9)


def _bb_width(close: np.ndarray, period: int = 20, mult: float = 2.0) -> np.ndarray:
    mid = _sma(close, period)
    std = pd.Series(close).rolling(period, min_periods=1).std(ddof=0).fillna(0).values
    return (2 * mult * std) / (mid + 1e-9)


def _macd_hist_norm(close: np.ndarray, fast: int = 12, slow: int = 26, sig: int = 9) -> np.ndarray:
    macd = _ema(close, fast) - _ema(close, slow)
    signal = _ema(macd, sig)
    hist = macd - signal
    atr = compute_atr(np.maximum(close, np.roll(close, 1)),
                      np.minimum(close, np.roll(close, 1)), close, 14)
    return hist / (atr + 1e-9)


def _rolling_zscore(x: np.ndarray, period: int) -> np.ndarray:
    series = pd.Series(x.astype(np.float64))
    mean = series.rolling(period, min_periods=2).mean()
    std = series.rolling(period, min_periods=2).std(ddof=0)
    return ((series - mean) / (std + 1e-9)).fillna(0.0).values


def _cci(high: np.ndarray, low: np.ndarray, close: np.ndarray, period: int = 14) -> np.ndarray:
    tp = (high + low + close) / 3.0
    sma = _sma(tp, period)
    mad = (pd.Series(tp).rolling(period).apply(lambda v: np.mean(np.abs(v - v.mean())), raw=True)
           .fillna(1e-9).values)
    return (tp - sma) / (0.015 * mad + 1e-9) / 200.0


def _parkinson_vol(high: np.ndarray, low: np.ndarray, period: int = 14) -> np.ndarray:
    log_hl = np.log((high + 1e-9) / (low + 1e-9)) ** 2
    factor = 1.0 / (4.0 * np.log(2))
    return np.sqrt(pd.Series(factor * log_hl).rolling(period, min_periods=1).mean().values)


def compute_atr(high: np.ndarray, low: np.ndarray, close: np.ndarray, period: int = 14) -> np.ndarray:
    tr = np.maximum(
        high[1:] - low[1:],
        np.maximum(np.abs(high[1:] - close[:-1]), np.abs(low[1:] - close[:-1])),
    )
    atr = np.zeros(len(close), dtype=np.float64)
    if period <= len(tr):
        atr[period] = tr[:period].mean()
        for i in range(period + 1, len(close)):
            atr[i] = (atr[i - 1] * (period - 1) + tr[i - 1]) / period
    return atr


FEATURE_NAMES = [
    "log_return",              # 0
    "norm_return",             # 1  log_ret / ATR14
    "close_position",          # 2  (C-L)/(H-L)
    "log_volume",              # 3  log(vol+1)
    "atr14_ratio",             # 4  ATR14/C
    "log_c_ema8",              # 5
    "log_c_ema21",             # 6
    "log_c_ema50",             # 7
    "log_ema8_ema21",          # 8
    "log_ema21_ema50",         # 9
    "rsi14",                   # 10
    "rsi7",                    # 11
    "bb_pct_b",                # 12
    "bb_width",                # 13
    "macd_hist_norm",          # 14
    "atr14_atr50",             # 15
    "parkinson_vol14",         # 16
    "vol_ratio_sma20",         # 17
    "sin_dow",                 # 18
    "cos_dow",                 # 19
    "sin_hod",                 # 20
    "cos_hod",                 # 21
    "lag_ret_1",               # 22
    "lag_ret_5",               # 23
    "lag_ret_20",              # 24
    "zscore_close_20",         # 25
    "zscore_close_50",         # 26
    "range_ratio",             # 27  (H-L)/C
    "zscore_range_20",         # 28
    "cci14",                   # 29
    "lag_norm_ret_2",          # 30
    "lag_norm_ret_3",          # 31
    "lag_norm_ret_5",          # 32
    "lag_norm_ret_8",          # 33
    "lag_norm_ret_13",         # 34
    "zscore_vol_20",           # 35
    "lag_rsi14_1",             # 36
    "lag_rsi14_3",             # 37
    "lag_bb_pct_b_1",          # 38
    "lag_bb_pct_b_3",          # 39
    "zscore_rsi14_20",         # 40
    "zscore_rsi7_20",          # 41
    "macd_hist_norm_dup",      # 42  (duplicate for MQL5 parity)
    "lag_macd_hist_1",         # 43
    "atr14_atr5",              # 44
    "zscore_atr14_20",         # 45
    "lag_ret_10",              # 46
    "lag_ret_15",              # 47
    "close_sma50_atr14",       # 48
    "close_sma200_atr14",      # 49
    "zscore_autocorr_1",       # 50
    "zscore_autocorr_5",       # 51
    "close_ema100_atr50",      # 52
    "zscore_vol_50",           # 53
    "atr50_atr14",             # 54
    "order_flow_imbalance",    # 55  (0 in Python, live in MQL5)
    "spike_time_norm",         # 56  (1 in Python, live in MQL5)
]
assert len(FEATURE_NAMES) == 57, f"Expected 57 feature names, got {len(FEATURE_NAMES)}"


def build_feature_matrix(
    open_: np.ndarray,
    high: np.ndarray,
    low: np.ndarray,
    close: np.ndarray,
    volume: np.ndarray,
    timestamps: Optional[pd.DatetimeIndex] = None,
) -> np.ndarray:
    """Build the canonical 57-feature matrix from OHLCV data.

    Matches AIFeatureVectorBuilder.mqh and data_pipeline.py exactly.
    Features 55 (OFI) and 56 (spike time) are zero/one placeholders
    since they require tick-level data only available at MT5 runtime.
    """
    n = len(close)
    log_ret = np.concatenate([[0.0], np.log(close[1:] / (close[:-1] + 1e-12))])
    atr14 = compute_atr(high, low, close, 14)
    atr50 = compute_atr(high, low, close, 50)
    atr5  = compute_atr(high, low, close, 5)

    cols = [
        log_ret,                                                          # 0
        log_ret / (atr14 + 1e-9),                                         # 1
        (close - low) / (high - low + 1e-9),                              # 2
        np.log(volume.astype(np.float64) + 1.0),                          # 3
        atr14 / (close + 1e-9),                                           # 4
        np.log(close / (_ema(close, 8) + 1e-9) + 1e-9),                   # 5
        np.log(close / (_ema(close, 21) + 1e-9) + 1e-9),                  # 6
        np.log(close / (_ema(close, 50) + 1e-9) + 1e-9),                  # 7
        np.log(_ema(close, 8) / (_ema(close, 21) + 1e-9) + 1e-9),         # 8
        np.log(_ema(close, 21) / (_ema(close, 50) + 1e-9) + 1e-9),        # 9
        _rsi(close, 14),                                                  # 10
        _rsi(close, 7),                                                   # 11
        _bb_pct_b(close, 20, 2.0),                                        # 12
        _bb_width(close, 20, 2.0),                                        # 13
        _macd_hist_norm(close, 12, 26, 9),                                # 14
        atr14 / (atr50 + 1e-9),                                           # 15
        _parkinson_vol(high, low, 14),                                    # 16
        volume / (_sma(volume.astype(np.float64), 20) + 1e-9),            # 17
        np.zeros(n, dtype=np.float64),                                    # 18  sin_dow placeholder
        np.zeros(n, dtype=np.float64),                                    # 19  cos_dow placeholder
        np.zeros(n, dtype=np.float64),                                    # 20  sin_hod placeholder
        np.zeros(n, dtype=np.float64),                                    # 21  cos_hod placeholder
        np.roll(log_ret, 1),                                              # 22
        np.roll(log_ret, 5),                                              # 23
        np.roll(log_ret, 20),                                             # 24
        _rolling_zscore(close.astype(np.float64), 20),                    # 25
        _rolling_zscore(close.astype(np.float64), 50),                    # 26
        (high - low) / (close + 1e-9),                                    # 27
        _rolling_zscore(high - low, 20),                                  # 28
        _cci(high, low, close, 14),                                       # 29
        np.roll(log_ret / (atr14 + 1e-9), 2),                            # 30
        np.roll(log_ret / (atr14 + 1e-9), 3),                            # 31
        np.roll(log_ret / (atr14 + 1e-9), 5),                            # 32
        np.roll(log_ret / (atr14 + 1e-9), 8),                            # 33
        np.roll(log_ret / (atr14 + 1e-9), 13),                           # 34
        _rolling_zscore(volume.astype(np.float64), 20),                   # 35
        np.roll(_rsi(close, 14), 1),                                      # 36
        np.roll(_rsi(close, 14), 3),                                      # 37
        np.roll(_bb_pct_b(close, 20, 2.0), 1),                           # 38
        np.roll(_bb_pct_b(close, 20, 2.0), 3),                           # 39
        _rolling_zscore(_rsi(close, 14), 20),                             # 40
        _rolling_zscore(_rsi(close, 7), 20),                              # 41
        _macd_hist_norm(close, 12, 26, 9),                                # 42  duplicate
        np.roll(_macd_hist_norm(close, 12, 26, 9), 1),                    # 43
        atr14 / (atr5 + 1e-9),                                            # 44
        _rolling_zscore(atr14, 20),                                       # 45
        np.roll(log_ret, 10),                                             # 46
        np.roll(log_ret, 15),                                             # 47
        (close - _sma(close, 50)) / (atr14 + 1e-9),                      # 48
        (close - _sma(close, 200)) / (atr14 + 1e-9),                     # 49
        _rolling_zscore(np.roll(log_ret, 1) * log_ret, 20),              # 50
        _rolling_zscore(np.roll(log_ret, 5) * log_ret, 20),              # 51
        (close - _ema(close, 100)) / (atr50 + 1e-9),                     # 52
        _rolling_zscore(volume.astype(np.float64), 50),                   # 53
        atr50 / (atr14 + 1e-9),                                           # 54
        np.zeros(n, dtype=np.float64),                                    # 55  OFI placeholder
        np.ones(n, dtype=np.float64),                                     # 56  spike time placeholder
    ]

    features = np.column_stack(cols).astype(np.float32)

    # Fill calendar features if timestamps provided
    if timestamps is not None:
        dow = timestamps.dayofweek.values / 6.0
        hod = timestamps.hour.values / 23.0
        features[:, 18] = np.sin(2 * np.pi * dow)
        features[:, 19] = np.cos(2 * np.pi * dow)
        features[:, 20] = np.sin(2 * np.pi * hod)
        features[:, 21] = np.cos(2 * np.pi * hod)

    # Sanitize: match MQL5 SanitizeFeature [-10, +10] clip
    features = np.nan_to_num(features, nan=0.0, posinf=3.0, neginf=-3.0)
    features = np.clip(features, -10.0, 10.0)

    return features


print(f"Feature engineering ready — {cfg.n_features} features defined")
print(f"Feature 55 (OFI) = 0.0 placeholder (requires tick data at runtime)")
print(f"Feature 56 (Spike) = 1.0 placeholder (requires tick data at runtime)")

## Cell 4 — Label Generation

Triple-barrier labeling with CUSUM event filtering.
Label encoding: **0=SELL, 1=NEUTRAL, 2=BUY** (matching MQL5 `OnnxBrain` signal mapping).

In [ ]:
# ============================================================
# Cell 4 — Label Generation
# ============================================================

def triple_barrier_labels(
    close: np.ndarray,
    high: np.ndarray,
    low: np.ndarray,
    atr: np.ndarray,
    k: float = 1.5,
    vertical_bars: int = 20,
) -> np.ndarray:
    """Triple-barrier labeling: +1=BUY, -1=SELL, 0=NEUTRAL."""
    n = len(close)
    labels = np.zeros(n, dtype=np.int8)
    for i in range(n - vertical_bars):
        if atr[i] <= 0:
            continue
        upper = close[i] + k * atr[i]
        lower = close[i] - k * atr[i]
        for j in range(i + 1, min(i + vertical_bars + 1, n)):
            if high[j] >= upper:
                labels[i] = 1
                break
            if low[j] <= lower:
                labels[i] = -1
                break
    return labels


def cusum_filter(
    close: np.ndarray,
    threshold_multiplier: float = 1.0,
    atr: Optional[np.ndarray] = None,
) -> List[int]:
    """CUSUM event filter — returns indices of significant moves."""
    if atr is None:
        atr = np.ones(len(close), dtype=np.float64)
    events: List[int] = []
    s_pos, s_neg = 0.0, 0.0
    for i in range(1, len(close)):
        ret = float(np.log(close[i] / (close[i - 1] + 1e-12)))
        thresh = max(1e-8, threshold_multiplier * (float(atr[i]) / (float(close[i]) + 1e-9)))
        s_pos = max(0.0, s_pos + ret)
        s_neg = min(0.0, s_neg + ret)
        if s_pos > thresh:
            events.append(i)
            s_pos = 0.0
        elif s_neg < -thresh:
            events.append(i)
            s_neg = 0.0
    return events


def compute_uniqueness_weights(
    event_indices: np.ndarray,
    vertical_bars: int,
    total_bars: int,
) -> np.ndarray:
    """Sample uniqueness weights — reduce weight on overlapping events."""
    if len(event_indices) == 0:
        return np.zeros(0, dtype=np.float32)
    concurrency = np.zeros(total_bars, dtype=np.float32)
    for idx in event_indices:
        end = min(int(idx) + vertical_bars, total_bars)
        concurrency[int(idx):end] += 1.0
    raw_weights = np.zeros(len(event_indices), dtype=np.float32)
    for i, idx in enumerate(event_indices):
        end = min(int(idx) + vertical_bars, total_bars)
        raw_weights[i] = np.mean(1.0 / np.maximum(concurrency[int(idx):end], 1e-9))
    total = raw_weights.sum()
    if total > 1e-9:
        raw_weights = raw_weights / total * len(event_indices)
    return raw_weights


def compute_forward_returns(
    prices: np.ndarray,
    event_indices: np.ndarray,
    holding_bars: int = 10,
) -> np.ndarray:
    """Forward returns for IC computation."""
    returns = np.zeros(len(event_indices), dtype=np.float32)
    for i, idx in enumerate(event_indices):
        end = min(int(idx) + holding_bars, len(prices) - 1)
        returns[i] = (prices[end] - prices[int(idx)]) / max(prices[int(idx)], 1e-10)
    return returns


# --- Build labels for each symbol ---
all_features_list = []
all_labels_list = []
all_weights_list = []
all_returns_list = []
all_timestamps_list = []

if cfg.symbol_col in df.columns:
    groups = df.sort_values([cfg.symbol_col, cfg.date_col]).groupby(cfg.symbol_col, sort=False)
else:
    groups = [("DEFAULT", df.sort_values(cfg.date_col))]

for sym, sym_df in groups:
    sym_df = sym_df.dropna(subset=["open", "high", "low", "close", "volume"]).copy()
    if len(sym_df) < max(cfg.seq_len + cfg.tb_vertical_bars + 20, 160):
        print(f"  Skipping {sym}: only {len(sym_df)} bars")
        continue

    close = sym_df["close"].to_numpy(dtype=np.float64)
    high  = sym_df["high"].to_numpy(dtype=np.float64)
    low   = sym_df["low"].to_numpy(dtype=np.float64)
    open_ = sym_df["open"].to_numpy(dtype=np.float64)
    vol   = sym_df["volume"].to_numpy(dtype=np.float64)
    ts    = sym_df[cfg.date_col].to_numpy()

    # Features
    if cfg.use_precomputed_features:
        feat_cols = sorted(
            [c for c in sym_df.columns if c.startswith("feature_")],
            key=lambda x: int(x.split("_")[1]),
        )
        features = sym_df[feat_cols].to_numpy(dtype=np.float32)
        features = np.nan_to_num(features, nan=0.0, posinf=3.0, neginf=-3.0)
        features = np.clip(features, -10.0, 10.0)
    else:
        features = build_feature_matrix(open_, high, low, close, vol,
                                        timestamps=pd.DatetimeIndex(sym_df[cfg.date_col]))

    # Labels
    atr = compute_atr(high, low, close, 14)
    raw_labels = triple_barrier_labels(close, high, low, atr, k=cfg.tb_k,
                                       vertical_bars=cfg.tb_vertical_bars)

    # CUSUM event filter
    events = np.asarray(cusum_filter(close, threshold_multiplier=cfg.cusum_threshold, atr=atr),
                        dtype=np.int32)
    label_indices = np.asarray([idx - 1 for idx in events if idx >= 1], dtype=np.int32)

    if len(label_indices) < 48:
        print(f"  Skipping {sym}: only {len(label_indices)} events")
        continue

    weights = compute_uniqueness_weights(label_indices, cfg.tb_vertical_bars, len(close))
    fwd_returns = compute_forward_returns(close, label_indices, holding_bars=cfg.tb_vertical_bars)

    all_features_list.append(features)
    all_labels_list.append(raw_labels)
    all_weights_list.append((label_indices, weights))
    all_returns_list.append((label_indices, fwd_returns))
    all_timestamps_list.append(ts)

    # Class distribution for this symbol
    sel_labels = raw_labels[label_indices]
    counts = {"SELL": (sel_labels == -1).sum(), "NEUTRAL": (sel_labels == 0).sum(),
              "BUY": (sel_labels == 1).sum()}
    print(f"  {sym}: {len(label_indices)} events | {counts}")

print(f"\nTotal symbols processed: {len(all_features_list)}")

## Cell 5 — Data Preprocessing

- StandardScaler (fit on train only)
- Sequence windowing (seq_len=60)
- Chronological train/val/test split
- Class weighting for imbalanced datasets

In [ ]:
# ============================================================
# Cell 5 — Data Preprocessing
# ============================================================

class TradingDataset(Dataset):
    def __init__(self, X, y, weights=None, returns=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.weights = torch.tensor(
            weights if weights is not None else np.ones(len(y), dtype=np.float32),
            dtype=torch.float32,
        )
        self.returns = torch.tensor(
            returns if returns is not None else np.zeros(len(y), dtype=np.float32),
            dtype=torch.float32,
        )

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.weights[idx], self.returns[idx]


def prepare_sequences(features, labels, weights, returns, timestamps, seq_len, sample_indices):
    """Build (seq_len, n_features) sequences from event-filtered indices.

    Label mapping: -1 → 0 (SELL), 0 → 1 (NEUTRAL), 1 → 2 (BUY)
    """
    X, y, w, r, ts = [], [], [], [], []
    for i, si in enumerate(sample_indices):
        if si < seq_len:
            continue
        end = int(si)
        X.append(features[end - seq_len:end])
        y.append(int(labels[end - 1]) + 1)   # -1→0, 0→1, 1→2
        w.append(float(weights[i]))
        r.append(float(returns[i]))
        ts.append(timestamps[end - 1])
    return (
        np.asarray(X, dtype=np.float32),
        np.asarray(y, dtype=np.int64),
        np.asarray(w, dtype=np.float32),
        np.asarray(r, dtype=np.float32),
        np.asarray(ts),
    )


def save_scaler_to_bin(scaler: StandardScaler, output_path: str) -> None:
    """Save scaler in MQL5-compatible binary format.

    Format: int32(n) + float64[n] means + float64[n] scales  (little-endian)
    """
    output = Path(output_path)
    output.parent.mkdir(parents=True, exist_ok=True)
    n = len(scaler.mean_)
    with output.open("wb") as f:
        f.write(struct.pack("<i", n))
        f.write(struct.pack(f"<{n}d", *scaler.mean_))
        f.write(struct.pack(f"<{n}d", *scaler.scale_))
    print(f"Scaler saved: {output} ({n} features)")


# --- Build per-symbol sequences, then concatenate ---
all_X, all_y, all_w, all_r, all_ts = [], [], [], [], []

for k_sym in range(len(all_features_list)):
    features = all_features_list[k_sym]
    raw_labels = all_labels_list[k_sym]
    label_indices, weights = all_weights_list[k_sym]
    _, fwd_returns = all_returns_list[k_sym]
    timestamps = all_timestamps_list[k_sym]

    X, y, w, r, ts = prepare_sequences(
        features, raw_labels, weights, fwd_returns, timestamps,
        seq_len=cfg.seq_len, sample_indices=label_indices,
    )
    if len(X) < 24:
        continue
    all_X.append(X)
    all_y.append(y)
    all_w.append(w)
    all_r.append(r)
    all_ts.append(ts)

X_all = np.concatenate(all_X)
y_all = np.concatenate(all_y)
w_all = np.concatenate(all_w)
r_all = np.concatenate(all_r)
ts_all = np.concatenate(all_ts)

print(f"Total sequences: {len(X_all)}")
print(f"X shape: {X_all.shape}")
print(f"Class distribution: SELL={np.sum(y_all==0)}, NEUTRAL={np.sum(y_all==1)}, BUY={np.sum(y_all==2)}")

# --- Chronological split ---
n = len(X_all)
train_end = int(n * cfg.train_ratio)
val_end   = int(n * (cfg.train_ratio + cfg.val_ratio))

X_train, y_train, w_train, r_train = X_all[:train_end], y_all[:train_end], w_all[:train_end], r_all[:train_end]
X_val,   y_val,   w_val,   r_val   = X_all[train_end:val_end], y_all[train_end:val_end], w_all[train_end:val_end], r_all[train_end:val_end]
X_test,  y_test,  w_test,  r_test  = X_all[val_end:], y_all[val_end:], w_all[val_end:], r_all[val_end:]

print(f"\nTrain: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

# --- StandardScaler (fit on train only, per-feature across all timesteps) ---
scaler = StandardScaler()
# Reshape to (n_samples * seq_len, n_features) so each of the 57 features
# gets a single mean/scale — matching the MQL5 EA's scaler.bin expectation.
scaler.fit(X_train.reshape(-1, cfg.n_features))

def apply_scaler(X, scaler, n_feat):
    orig_shape = X.shape
    flat = X.reshape(-1, n_feat)
    scaled = scaler.transform(flat)
    return scaled.reshape(orig_shape).astype(np.float32)

X_train = apply_scaler(X_train, scaler, cfg.n_features)
X_val   = apply_scaler(X_val,   scaler, cfg.n_features)
X_test  = apply_scaler(X_test,  scaler, cfg.n_features)

# Save scaler
scaler_path = str(Path(cfg.output_dir) / "scaler.bin")
save_scaler_to_bin(scaler, scaler_path)

# --- Class weights ---
counts = np.bincount(y_train, minlength=cfg.n_classes)
class_weights = 1.0 / np.maximum(counts, 1)
class_weights = class_weights / class_weights.sum() * cfg.n_classes  # normalize
print(f"\nClass weights: {dict(zip(['SELL','NEUTRAL','BUY'], class_weights.round(3)))}")

# --- DataLoaders ---
train_ds = TradingDataset(X_train, y_train, w_train, r_train)
val_ds   = TradingDataset(X_val,   y_val,   w_val,   r_val)
test_ds  = TradingDataset(X_test,  y_test,  w_test,  r_test)

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,  drop_last=False)
val_loader   = DataLoader(val_ds,   batch_size=cfg.batch_size, shuffle=False, drop_last=False)
test_loader  = DataLoader(test_ds,  batch_size=cfg.batch_size, shuffle=False, drop_last=False)

print(f"\nDataLoaders ready — train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}")

## Cell 6 — Model Definitions

Five model architectures, all sharing the same interface:
- **Input**: `(B, 60, 57)`
- **Output**: `(B, 3)` raw logits

| Model | Description |
|-------|-------------|
| SequenceMLP | Baseline MLP on flattened sequence |
| PatchTST | Patch-based Transformer (SOTA for TS) |
| Conv1D | 1D-CNN feature extraction + FC head |
| LSTM | Classic recurrent model |
| LightGBM | Gradient boosting on flattened features |

In [ ]:
# ============================================================
# Cell 6 — Model Definitions
# ============================================================

N_FEATURES = cfg.n_features
N_CLASSES  = cfg.n_classes


# --- Model A: SequenceMLP ---
class SequenceMLP(nn.Module):
    """Export-safe baseline model for MT5 ONNX runtime."""
    def __init__(self, seq_len=60, n_features=N_FEATURES,
                 hidden1=256, hidden2=128, dropout=0.10, n_classes=N_CLASSES):
        super().__init__()
        in_features = seq_len * n_features
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.LayerNorm(in_features),
            nn.Linear(in_features, hidden1),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden1, hidden2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden2, n_classes),
        )

    def forward(self, x):
        return self.net(x)


# --- Model B: PatchTST ---
class PatchTST(nn.Module):
    """Patch-based Transformer for financial time series."""
    def __init__(self, seq_len=60, n_features=N_FEATURES,
                 patch_len=12, stride=6,
                 d_model=128, n_heads=8, n_layers=3,
                 dropout=0.1, n_classes=N_CLASSES):
        super().__init__()
        self.patch_len = patch_len
        self.stride = stride
        self.n_patches = (seq_len - patch_len) // stride + 1

        self.patch_embed = nn.Linear(patch_len, d_model)
        self.cls_token = nn.Parameter(torch.zeros(1, n_features, 1, d_model))
        self.pos_embed = nn.Parameter(torch.zeros(1, n_features, self.n_patches + 1, d_model))
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=dropout, norm_first=True,
            batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(n_features * d_model, n_classes)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        B, S, F = x.shape
        x = x.permute(0, 2, 1)                          # (B, F, S)
        patches = x.unfold(-1, self.patch_len, self.stride)  # (B, F, n_patches, patch_len)
        patches = self.patch_embed(patches)               # (B, F, n_patches, d_model)
        cls = self.cls_token.expand(B, -1, -1, -1)
        patches = torch.cat([cls, patches], dim=2) + self.pos_embed
        b2, f2, np_, dm = patches.shape
        patches = self.transformer(patches.reshape(b2 * f2, np_, dm))
        patches = patches.reshape(b2, f2, np_, dm)
        out = self.norm(patches[:, :, 0, :]).reshape(B, -1)  # CLS token
        return self.head(self.drop(out))


# --- Model C: Conv1D ---
class Conv1DModel(nn.Module):
    """1D-CNN feature extraction + FC head."""
    def __init__(self, seq_len=60, n_features=N_FEATURES,
                 hidden=128, dropout=0.1, n_classes=N_CLASSES):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(n_features, 64, kernel_size=3, padding=1),
            nn.GELU(),
            nn.BatchNorm1d(64),
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.GELU(),
            nn.BatchNorm1d(128),
            nn.AdaptiveAvgPool1d(1),
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, n_classes),
        )

    def forward(self, x):
        x = x.permute(0, 2, 1)   # (B, F, S)
        x = self.conv(x)
        return self.head(x)


# --- Model D: LSTM ---
class LSTMModel(nn.Module):
    """LSTM sequence model."""
    def __init__(self, seq_len=60, n_features=N_FEATURES,
                 hidden_size=128, num_layers=2, dropout=0.1, n_classes=N_CLASSES):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.head = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, n_classes),
        )

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)
        return self.head(h_n[-1])   # last layer hidden state


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# --- Instantiate all models ---
model_defs = {
    "SequenceMLP": lambda: SequenceMLP(seq_len=cfg.seq_len, n_features=cfg.n_features),
    "PatchTST":    lambda: PatchTST(seq_len=cfg.seq_len, n_features=cfg.n_features),
    "Conv1D":      lambda: Conv1DModel(seq_len=cfg.seq_len, n_features=cfg.n_features),
    "LSTM":        lambda: LSTMModel(seq_len=cfg.seq_len, n_features=cfg.n_features),
}

print("Model architectures:")
for name, factory in model_defs.items():
    m = factory()
    dummy = torch.zeros(1, cfg.seq_len, cfg.n_features)
    out = m(dummy)
    print(f"  {name:15s} | params={count_parameters(m):>10,} | output={tuple(out.shape)}")
    del m

print("\nLightGBM will be trained separately on flattened features.")

## Cell 7 — Training Loop

PyTorch training with:
- AdamW + cosine annealing LR
- CrossEntropyLoss with class weights
- Early stopping on validation IC
- Gradient clipping (max_norm=1.0)
- Mixed precision (AMP) if GPU available

In [ ]:
# ============================================================
# Cell 7 — Training Loop
# ============================================================

def compute_ic(model, loader, device):
    """Information Coefficient: Spearman corr between predicted score and forward returns."""
    model.eval()
    scores, returns = [], []
    with torch.no_grad():
        for x, _y, _w, ret in loader:
            probs = torch.softmax(model(x.to(device)), dim=-1).cpu().numpy()
            scores.extend((probs[:, 2] - probs[:, 0]).tolist())  # BUY - SELL
            returns.extend(ret.numpy().tolist())
    ic, _ = spearmanr(scores, returns)
    return float(ic) if not np.isnan(ic) else 0.0


def compute_metrics(model, loader, device):
    """Compute accuracy, macro F1, per-class precision/recall."""
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x, y, _w, _r in loader:
            logits = model(x.to(device))
            preds = logits.argmax(dim=-1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(y.numpy())
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    return {
        "accuracy": accuracy_score(all_labels, all_preds),
        "f1_macro": f1_score(all_labels, all_preds, average="macro", zero_division=0),
        "precision_per_class": precision_score(all_labels, all_preds, average=None, zero_division=0).tolist(),
        "recall_per_class": recall_score(all_labels, all_preds, average=None, zero_division=0).tolist(),
        "confusion": confusion_matrix(all_labels, all_preds),
    }


def train_model(
    name: str,
    model: nn.Module,
    train_loader,
    val_loader,
    class_weights_np: np.ndarray,
    device: str = DEVICE,
    epochs: int = cfg.epochs,
    lr: float = cfg.lr,
    weight_decay: float = cfg.weight_decay,
    patience: int = cfg.patience,
    grad_clip: float = cfg.grad_clip,
    use_amp: bool = cfg.amp,
):
    """Train a single model with early stopping on validation IC."""
    model = model.to(device)
    cw = torch.tensor(class_weights_np, dtype=torch.float32, device=device)
    criterion = nn.CrossEntropyLoss(weight=cw, reduction="none")
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    total_steps = max(1, epochs * len(train_loader))
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, total_steps=total_steps, pct_start=0.15,
    )
    scaler_amp = torch.cuda.amp.GradScaler() if (use_amp and device == "cuda") else None

    best_state = None
    best_val_ic = -1e9
    wait = 0
    history = {"train_loss": [], "val_ic": [], "val_acc": []}

    print(f"\n{'='*60}")
    print(f"Training: {name} | params={count_parameters(model):,}")
    print(f"{'='*60}")

    for epoch in range(epochs):
        # --- Train ---
        model.train()
        epoch_loss = 0.0
        for x, y, w, _r in train_loader:
            x, y, w = x.to(device), y.to(device), w.to(device)
            optimizer.zero_grad()
            if scaler_amp is not None:
                with torch.cuda.amp.autocast():
                    loss = (criterion(model(x), y) * w).mean()
                scaler_amp.scale(loss).backward()
                scaler_amp.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                scaler_amp.step(optimizer)
                scaler_amp.update()
            else:
                loss = (criterion(model(x), y) * w).mean()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                optimizer.step()
            scheduler.step()
            epoch_loss += loss.item()

        epoch_loss /= max(1, len(train_loader))

        # --- Validate ---
        val_ic = compute_ic(model, val_loader, device)
        val_metrics = compute_metrics(model, val_loader, device)

        history["train_loss"].append(epoch_loss)
        history["val_ic"].append(val_ic)
        history["val_acc"].append(val_metrics["accuracy"])

        if val_ic > best_val_ic:
            best_val_ic = val_ic
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1

        if (epoch + 1) % 5 == 0 or epoch == 0 or wait == 0:
            print(f"  epoch={epoch+1:03d} loss={epoch_loss:.5f} "
                  f"val_ic={val_ic:.4f} val_acc={val_metrics['accuracy']:.4f} "
                  f"best_ic={best_val_ic:.4f} patience={wait}/{patience}")

        if wait >= patience:
            print(f"  Early stopping at epoch {epoch+1}")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    model = model.to(device)

    # Final metrics
    final_ic = compute_ic(model, val_loader, device)
    final_metrics = compute_metrics(model, val_loader, device)
    print(f"  FINAL: val_ic={final_ic:.4f} val_acc={final_metrics['accuracy']:.4f} "
          f"f1_macro={final_metrics['f1_macro']:.4f}")

    return model, best_val_ic, history, final_metrics


print("Training loop ready.")

## Cell 8 — Model Comparison

Train all 5 models (4 PyTorch + 1 LightGBM) and compare on the validation set.

In [ ]:
# ============================================================
# Cell 8 — Model Comparison
# ============================================================

results = {}
trained_models = {}

# --- Train PyTorch models ---
for name, factory in model_defs.items():
    set_seed(cfg.seed)
    model = factory()
    trained, best_ic, hist, metrics = train_model(
        name=name,
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        class_weights_np=class_weights,
    )
    results[name] = {
        "val_ic": best_ic,
        "accuracy": metrics["accuracy"],
        "f1_macro": metrics["f1_macro"],
        "history": hist,
        "metrics": metrics,
    }
    trained_models[name] = trained

# --- Train LightGBM ---
try:
    import lightgbm as lgb

    X_tr_flat = X_train.reshape(len(X_train), -1)
    X_va_flat = X_val.reshape(len(X_val), -1)

    lgb_model = lgb.train(
        {
            "objective": "multiclass",
            "num_class": 3,
            "metric": "multi_logloss",
            "learning_rate": 0.03,
            "num_leaves": 63,
            "feature_fraction": 0.7,
            "bagging_fraction": 0.8,
            "bagging_freq": 5,
            "lambda_l1": 0.1,
            "lambda_l2": 0.1,
            "min_child_samples": 50,
            "verbose": -1,
        },
        lgb.Dataset(X_tr_flat, label=y_train),
        valid_sets=[lgb.Dataset(X_va_flat, label=y_val)],
        num_boost_round=1000,
        callbacks=[lgb.early_stopping(50)],
    )

    lgb_preds = lgb_model.predict(X_va_flat)
    lgb_labels = lgb_preds.argmax(axis=1)
    lgb_score = lgb_preds[:, 2] - lgb_preds[:, 0]
    lgb_ic, _ = spearmanr(lgb_score, r_val)
    lgb_ic = float(lgb_ic) if not np.isnan(lgb_ic) else 0.0
    lgb_acc = accuracy_score(y_val, lgb_labels)
    lgb_f1 = f1_score(y_val, lgb_labels, average="macro", zero_division=0)

    results["LightGBM"] = {
        "val_ic": lgb_ic,
        "accuracy": lgb_acc,
        "f1_macro": lgb_f1,
        "metrics": {
            "accuracy": lgb_acc,
            "f1_macro": lgb_f1,
            "confusion": confusion_matrix(y_val, lgb_labels),
        },
    }
    trained_models["LightGBM"] = lgb_model
    print(f"\nLightGBM: val_ic={lgb_ic:.4f} val_acc={lgb_acc:.4f} f1_macro={lgb_f1:.4f}")

except ImportError:
    print("LightGBM not available, skipping.")

# --- Comparison table ---
print("\n" + "="*70)
print(f"{'Model':15s} | {'Val IC':>8s} | {'Accuracy':>8s} | {'F1 Macro':>8s}")
print("-" * 70)
for name, r in sorted(results.items(), key=lambda x: x[1]["val_ic"], reverse=True):
    print(f"{name:15s} | {r['val_ic']:8.4f} | {r['accuracy']:8.4f} | {r['f1_macro']:8.4f}")
print("="*70)

# --- Select best model ---
best_name = max(results, key=lambda k: results[k]["val_ic"])
print(f"\nBest model: {best_name} (val_ic={results[best_name]['val_ic']:.4f})")

# --- Plot training curves ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for name, r in results.items():
    if "history" not in r:
        continue
    h = r["history"]
    axes[0].plot(h["train_loss"], label=name)
    axes[1].plot(h["val_ic"], label=name)
    axes[2].plot(h["val_acc"], label=name)

axes[0].set_title("Training Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()
axes[1].set_title("Validation IC"); axes[1].set_xlabel("Epoch"); axes[1].legend()
axes[2].set_title("Validation Accuracy"); axes[2].set_xlabel("Epoch"); axes[2].legend()
plt.tight_layout()
plt.savefig(str(Path(cfg.output_dir) / "training_curves.png"), dpi=150)
plt.show()

# --- Confusion matrices ---
class_names = ["SELL", "NEUTRAL", "BUY"]
n_models = len([k for k in results if "metrics" in results[k]])
fig, axes = plt.subplots(1, max(n_models, 1), figsize=(5 * n_models, 4))
if n_models == 1:
    axes = [axes]
for ax, (name, r) in zip(axes, [(k, v) for k, v in results.items() if "metrics" in v]):
    cm = r["metrics"]["confusion"]
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=class_names, yticklabels=class_names)
    ax.set_title(f"{name}\nIC={r['val_ic']:.3f}")
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
plt.tight_layout()
plt.savefig(str(Path(cfg.output_dir) / "confusion_matrices.png"), dpi=150)
plt.show()

## Cell 9 — CPCV Validation

Combinatorial Purged Cross-Validation:
- 6 groups, 2 test groups → 15 combinations
- Embargo period between train/test
- PSR (Probabilistic Sharpe Ratio) deployment gate
- Deploy only if PSR > 0.95

In [ ]:
# ============================================================
# Cell 9 — CPCV Validation
# ============================================================

from itertools import combinations


def purge_embargo(train_idx, test_idx, purge=5, embargo=5):
    """Remove train samples within purge/embargo window of test set."""
    t0, t1 = test_idx.min(), test_idx.max()
    mask = (train_idx < t0 - purge) | (train_idx > t1 + embargo)
    return train_idx[mask]


def cpcv_folds(n, n_splits=6, n_test=2, purge=5, embargo=5):
    """Generate CPCV fold indices."""
    groups = np.array_split(np.arange(n), n_splits)
    folds = []
    for test_groups in combinations(range(n_splits), n_test):
        test_idx = np.concatenate([groups[i] for i in test_groups])
        train_idx = np.concatenate([groups[i] for i in range(n_splits) if i not in test_groups])
        train_idx = purge_embargo(train_idx, test_idx, purge, embargo)
        folds.append((train_idx, test_idx))
    return folds


def psr(sharpe_ratios, sr_ref=0.0):
    """Probabilistic Sharpe Ratio."""
    mu = sharpe_ratios.mean()
    sig = sharpe_ratios.std(ddof=1) + 1e-9
    z = (mu - sr_ref) / sig * np.sqrt(len(sharpe_ratios))
    return float(norm.cdf(z))


def estimate_annualization(timestamps):
    """Estimate annualization factor from timestamps."""
    if len(timestamps) < 3:
        return 252.0
    ts = pd.to_datetime(pd.Series(timestamps)).astype("int64") // 10**9
    deltas = np.diff(ts.to_numpy(dtype=np.int64))
    deltas = deltas[deltas > 0]
    if len(deltas) == 0:
        return 252.0
    median_seconds = float(np.median(deltas))
    if median_seconds <= 0:
        return 252.0
    return max(1.0, (365.25 * 24.0 * 3600.0) / median_seconds)


annualization = estimate_annualization(ts_all)
print(f"Annualization factor: {annualization:.1f}")

# --- Run CPCV on the best PyTorch model ---
best_torch_name = best_name if best_name != "LightGBM" else list(model_defs.keys())[0]
best_model = trained_models[best_torch_name]

# Export to ONNX temporarily for CPCV
tmp_onnx = str(Path(cfg.output_dir) / "_cpcv_tmp.onnx")
best_model.eval()
torch.onnx.export(
    best_model,
    torch.zeros(1, cfg.seq_len, cfg.n_features, dtype=torch.float32),
    tmp_onnx,
    opset_version=cfg.onnx_opset,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={"input": {0: "batch"}, "output": {0: "batch"}},
    do_constant_folding=True,
)

session = ort.InferenceSession(tmp_onnx)
input_name = session.get_inputs()[0].name

# Combine all data for CPCV
X_cpcv = np.concatenate([X_train, X_val, X_test], axis=0).astype(np.float32)
y_cpcv = np.concatenate([y_train, y_val, y_test])
r_cpcv = np.concatenate([r_train, r_val, r_test])

folds = cpcv_folds(len(X_cpcv), n_splits=cfg.cpcv_n_splits, n_test=cfg.cpcv_n_test,
                   purge=cfg.cpcv_purge, embargo=cfg.cpcv_embargo)

print(f"\nCPCV: {len(folds)} folds ({cfg.cpcv_n_splits} groups, {cfg.cpcv_n_test} test)")

sharpe_ratios = []
for i, (train_idx, test_idx) in enumerate(folds):
    X_test_fold = X_cpcv[test_idx]
    logits = session.run(None, {input_name: X_test_fold})[0]
    preds = logits.argmax(axis=1) - 1   # 0,1,2 → -1,0,1
    returns = preds * r_cpcv[test_idx]
    sr_val = returns.mean() / (returns.std() + 1e-9) * np.sqrt(annualization)
    sharpe_ratios.append(float(sr_val))

sharpe_ratios = np.asarray(sharpe_ratios, dtype=np.float64)
p10_sr = float(np.percentile(sharpe_ratios, 10))
psr_val = psr(sharpe_ratios)
deploy_gate = p10_sr > 0.0 and psr_val > cfg.psr_threshold

print(f"\nCPCV Results ({len(folds)} folds):")
print(f"  Sharpe per fold: {np.round(sharpe_ratios, 3)}")
print(f"  Mean SR:  {sharpe_ratios.mean():.3f}")
print(f"  10th pct SR: {p10_sr:.3f}")
print(f"  PSR: {psr_val:.3f}")
print(f"  DEPLOYMENT GATE: {'PASS' if deploy_gate else 'FAIL'}")

# Plot Sharpe distribution
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(len(sharpe_ratios)), sharpe_ratios, color=["green" if s > 0 else "red" for s in sharpe_ratios])
ax.axhline(y=0, color="black", linestyle="--", linewidth=0.5)
ax.axhline(y=p10_sr, color="orange", linestyle="--", linewidth=1, label=f"P10 SR={p10_sr:.3f}")
ax.set_xlabel("Fold"); ax.set_ylabel("Sharpe Ratio"); ax.set_title("CPCV Sharpe Ratios")
ax.legend()
plt.tight_layout()
plt.savefig(str(Path(cfg.output_dir) / "cpcv_sharpe.png"), dpi=150)
plt.show()

# Cleanup temp file
Path(tmp_onnx).unlink(missing_ok=True)

## Cell 10 — ONNX Export

Export the best model to ONNX format with verification:
- ONNX output matches PyTorch output (max diff < 1e-5)
- Save `model.onnx` with correct input/output shapes
- Save `scaler.bin` in MQL5-compatible format
- Save training metadata as JSON

In [ ]:
# ============================================================
# Cell 10 — ONNX Export
# ============================================================

def export_onnx_model(model, seq_len, n_feat, output_path, opset=12):
    """Export PyTorch model to ONNX with shape verification."""
    model.eval()
    dummy = torch.zeros(1, seq_len, n_feat, dtype=torch.float32)
    torch.onnx.export(
        model,
        dummy,
        output_path,
        opset_version=opset,
        input_names=["input"],
        output_names=["output"],
        dynamic_axes={"input": {0: "batch"}, "output": {0: "batch"}},
        do_constant_folding=True,
    )

def export_lgbm_onnx(lgb_model, n_flat_features, output_path, opset=12):
    """Export LightGBM model to ONNX via onnxmltools."""
    try:
        import onnxmltools
        from onnxmltools.convert.common.data_types import FloatTensorType
        onnx_model = onnxmltools.convert_lightgbm(
            lgb_model,
            name="lgbm_trading",
            initial_types=[("float_input", FloatTensorType([None, n_flat_features]))],
            target_opset=opset,
        )
        onnxmltools.utils.save_model(onnx_model, output_path)
        return True
    except ImportError:
        print("onnxmltools not installed; LightGBM ONNX export skipped.")
        return False


def verify_onnx(onnx_path, model, seq_len, n_feat, n_samples=5, tol=1e-5):
    """Verify ONNX output matches PyTorch output."""
    session = ort.InferenceSession(onnx_path)
    inp_name = session.get_inputs()[0].name
    model.eval()
    max_diff = 0.0
    for _ in range(n_samples):
        x = torch.randn(1, seq_len, n_feat, dtype=torch.float32)
        with torch.no_grad():
            pt_out = model(x).numpy()
        ort_out = session.run(None, {inp_name: x.numpy()})[0]
        diff = np.max(np.abs(pt_out - ort_out))
        max_diff = max(max_diff, diff)
    passed = max_diff < tol
    print(f"  ONNX verification: max_diff={max_diff:.2e} | {'PASS' if passed else 'FAIL'} (tol={tol:.0e})")
    return passed


# --- Export best model ---
onnx_path = str(Path(cfg.output_dir) / "model.onnx")

if best_name == "LightGBM":
    n_flat = cfg.seq_len * cfg.n_features
    export_lgbm_onnx(trained_models["LightGBM"], n_flat, onnx_path, cfg.onnx_opset)
    print(f"Exported LightGBM → {onnx_path}")
    # LightGBM ONNX has different input shape (B, seq_len*n_feat), not (B, seq_len, n_feat)
    # The EA expects (1, 60, 57), so LightGBM is NOT directly compatible.
    # If LightGBM wins, we should train the best PyTorch model instead.
    print("WARNING: LightGBM ONNX uses flattened input (B, 3420), not (B, 60, 57).")
    print("Falling back to best PyTorch model for EA compatibility.")
    best_torch_name = max(
        {k: v for k, v in results.items() if k != "LightGBM"},
        key=lambda k: results[k]["val_ic"],
    )
    best_name = best_torch_name
    best_model = trained_models[best_name]

export_onnx_model(best_model, cfg.seq_len, cfg.n_features, onnx_path, cfg.onnx_opset)
print(f"\nExported {best_name} → {onnx_path}")

# Verify
verify_onnx(onnx_path, best_model, cfg.seq_len, cfg.n_features)

# Check ONNX model shapes
onnx_model = onnx.load(onnx_path)
inp = onnx_model.graph.input[0]
out = onnx_model.graph.output[0]
inp_dims = [d.dim_value for d in inp.type.tensor_type.shape.dim]
out_dims = [d.dim_value for d in out.type.tensor_type.shape.dim]
# Note: batch dim may show 0 (dynamic) due to dynamic_axes — this is expected
print(f"  ONNX input:  {inp_dims}  (batch dim 0 = dynamic)")
print(f"  ONNX output: {out_dims}  (batch dim 0 = dynamic)")
assert inp_dims[1:] == [cfg.seq_len, cfg.n_features], f"Unexpected input shape: {inp_dims}"
assert out_dims[-1] == cfg.n_classes, f"Unexpected output shape: {out_dims}"

# --- Save training metadata ---
metadata = {
    "model_name": best_name,
    "seq_len": cfg.seq_len,
    "n_features": cfg.n_features,
    "n_classes": cfg.n_classes,
    "class_names": ["SELL", "NEUTRAL", "BUY"],
    "val_ic": float(results[best_name]["val_ic"]),
    "val_accuracy": float(results[best_name]["accuracy"]),
    "val_f1_macro": float(results[best_name]["f1_macro"]),
    "cpcv_mean_sr": float(sharpe_ratios.mean()),
    "cpcv_p10_sr": float(p10_sr),
    "cpcv_psr": float(psr_val),
    "deploy_gate": bool(deploy_gate),
    "annualization": float(annualization),
    "scaler_path": "scaler.bin",
    "onnx_opset": cfg.onnx_opset,
    "training_config": {
        "epochs": cfg.epochs,
        "batch_size": cfg.batch_size,
        "lr": cfg.lr,
        "weight_decay": cfg.weight_decay,
        "tb_k": cfg.tb_k,
        "tb_vertical_bars": cfg.tb_vertical_bars,
        "seed": cfg.seed,
    },
    "feature_names": FEATURE_NAMES,
}

metadata_path = str(Path(cfg.output_dir) / "training_metadata.json")
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)
print(f"\nMetadata saved: {metadata_path}")

print("\n" + "="*60)
print("EXPORT SUMMARY")
print("="*60)
print(f"  model.onnx    : {onnx_path}")
print(f"  scaler.bin    : {scaler_path}")
print(f"  metadata.json : {metadata_path}")
print(f"  Best model    : {best_name}")
print(f"  Deploy gate   : {'PASS' if deploy_gate else 'FAIL'}")

## Cell 11 — Model Interpretability

- Permutation feature importance
- SHAP values for top features
- Attention heatmap (if Transformer-based)
- Prediction distribution analysis

In [ ]:
# ============================================================
# Cell 11 — Model Interpretability
# ============================================================

# --- Permutation Importance ---
print("Computing permutation feature importance...")

best_model.eval()
baseline_ic = compute_ic(best_model, val_loader, DEVICE)
importance = np.zeros(cfg.n_features, dtype=np.float64)

# Permute each feature column across all timesteps
X_val_perm = X_val.copy()
for feat_idx in range(cfg.n_features):
    X_perm = X_val_perm.copy()
    np.random.shuffle(X_perm[:, :, feat_idx])  # permute across samples
    perm_ds = TradingDataset(X_perm, y_val, w_val, r_val)
    perm_loader = DataLoader(perm_ds, batch_size=cfg.batch_size, shuffle=False, drop_last=False)
    perm_ic = compute_ic(best_model, perm_loader, DEVICE)
    importance[feat_idx] = baseline_ic - perm_ic  # positive = important

# Plot top 20 features
top_k = 20
top_indices = np.argsort(importance)[::-1][:top_k]
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(range(top_k), importance[top_indices][::-1], color="steelblue")
ax.set_yticks(range(top_k))
ax.set_yticklabels([FEATURE_NAMES[i] for i in top_indices][::-1])
ax.set_xlabel("IC Drop (Permutation Importance)")
ax.set_title(f"Top {top_k} Features — {best_name}")
plt.tight_layout()
plt.savefig(str(Path(cfg.output_dir) / "permutation_importance.png"), dpi=150)
plt.show()

print(f"\nTop 10 features:")
for i, idx in enumerate(top_indices[:10]):
    print(f"  {i+1:2d}. {FEATURE_NAMES[idx]:30s} importance={importance[idx]:.4f}")


# --- SHAP values (for LightGBM or small PyTorch models) ---
try:
    import shap

    if "LightGBM" in trained_models:
        print("\nComputing SHAP values for LightGBM...")
        X_tr_flat = X_train.reshape(len(X_train), -1)[:500]  # subsample for speed
        explainer = shap.TreeExplainer(trained_models["LightGBM"])
        shap_values = explainer.shap_values(X_tr_flat)

        # SHAP summary plot for BUY class (index 2)
        flat_feat_names = [f"{FEATURE_NAMES[f]}_t{t}" for t in range(cfg.seq_len) for f in range(cfg.n_features)]
        shap.summary_plot(shap_values[2], X_tr_flat, feature_names=flat_feat_names, max_display=20, show=False)
        plt.title("SHAP — BUY class")
        plt.tight_layout()
        plt.savefig(str(Path(cfg.output_dir) / "shap_summary.png"), dpi=150)
        plt.show()
    else:
        print("LightGBM not trained; skipping SHAP (DeepExplainer is slow on Kaggle).")
except ImportError:
    print("SHAP not available, skipping.")


# --- Attention Heatmap (PatchTST only) ---
if best_name == "PatchTST":
    print("\nExtracting PatchTST attention weights...")
    best_model.eval()
    # Run a single sample through the model to get attention
    sample_x = torch.tensor(X_val[:1], dtype=torch.float32).to(DEVICE)
    with torch.no_grad():
        # Manually compute patches for visualization
        x = sample_x.permute(0, 2, 1)
        patches = x.unfold(-1, best_model.patch_len, best_model.stride)
        patches = best_model.patch_embed(patches)
        cls = best_model.cls_token.expand(1, -1, -1, -1)
        patches = torch.cat([cls, patches], dim=2) + best_model.pos_embed
        b2, f2, np_, dm = patches.shape
        # Run through transformer layers to get attention
        attn_weights = []
        x_in = patches.reshape(b2 * f2, np_, dm)
        for layer in best_model.transformer.layers:
            # Extract self-attention weights
            x_norm = layer.norm1(x_in)
            attn_out, attn_w = layer.self_attn(
                x_norm, x_norm, x_norm, need_weights=True, average_attn_weights=True
            )
            attn_weights.append(attn_w.mean(dim=0).cpu().numpy())  # avg over features
            x_in = x_in + layer.dropout1(attn_out)
            x_in = x_in + layer.dropout2(layer.linear2(layer.activation(layer.linear1(layer.norm2(x_in)))))

    # Plot attention from last layer (CLS token attending to patches)
    last_attn = attn_weights[-1]  # (n_patches+1, n_patches+1)
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(last_attn, cmap="YlOrRd", ax=ax,
                xticklabels=["CLS"] + [f"P{i}" for i in range(last_attn.shape[1]-1)],
                yticklabels=["CLS"] + [f"P{i}" for i in range(last_attn.shape[0]-1)])
    ax.set_title("PatchTST Attention (Last Layer, Feature-Averaged)")
    plt.tight_layout()
    plt.savefig(str(Path(cfg.output_dir) / "attention_heatmap.png"), dpi=150)
    plt.show()


# --- Prediction Distribution ---
best_model.eval()
all_probs = []
with torch.no_grad():
    for x, _y, _w, _r in val_loader:
        probs = torch.softmax(best_model(x.to(DEVICE)), dim=-1).cpu().numpy()
        all_probs.append(probs)
all_probs = np.concatenate(all_probs)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, (name, color) in enumerate(zip(["SELL", "NEUTRAL", "BUY"], ["#e74c3c", "#95a5a6", "#2ecc71"])):
    axes[i].hist(all_probs[:, i], bins=50, color=color, alpha=0.7, edgecolor="black")
    axes[i].set_title(f"P({name})")
    axes[i].set_xlabel("Probability")
    axes[i].axvline(x=0.5, color="black", linestyle="--", linewidth=0.5)
plt.suptitle(f"Prediction Distribution — {best_name}")
plt.tight_layout()
plt.savefig(str(Path(cfg.output_dir) / "prediction_distribution.png"), dpi=150)
plt.show()

## Cell 12 — Backtest Simulation

Simple vectorized backtest using model predictions:
- Sharpe ratio, max drawdown, win rate, profit factor
- Equity curve
- Compare against buy-and-hold baseline

In [ ]:
# ============================================================
# Cell 12 — Backtest Simulation
# ============================================================

best_model.eval()
all_preds = []
all_probs_bt = []
with torch.no_grad():
    for x, _y, _w, _r in test_loader:
        logits = best_model(x.to(DEVICE))
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
        all_preds.extend(probs.argmax(axis=1))
        all_probs_bt.append(probs)

all_preds = np.array(all_preds)
all_probs_bt = np.concatenate(all_probs_bt)

# Convert predictions to positions: 0=SELL(-1), 1=NEUTRAL(0), 2=BUY(+1)
positions = all_preds - 1  # -1, 0, +1

# Use forward returns as proxy for P&L
strategy_returns = positions * r_test

# --- Metrics ---
cumulative = np.cumprod(1 + strategy_returns)
running_max = np.maximum.accumulate(cumulative)
drawdown = (cumulative - running_max) / running_max
max_dd = float(drawdown.min())

mean_ret = strategy_returns.mean()
std_ret = strategy_returns.std() + 1e-9
sharpe = mean_ret / std_ret * np.sqrt(annualization)

win_mask = strategy_returns > 0
loss_mask = strategy_returns < 0
win_rate = win_mask.sum() / max(win_mask.sum() + loss_mask.sum(), 1)
profit_factor = strategy_returns[win_mask].sum() / max(-strategy_returns[loss_mask].sum(), 1e-9)

# Buy-and-hold baseline
bh_returns = r_test  # long-only
bh_cumulative = np.cumprod(1 + bh_returns)
bh_sharpe = bh_returns.mean() / (bh_returns.std() + 1e-9) * np.sqrt(annualization)

print("="*50)
print("BACKTEST RESULTS (Test Set)")
print("="*50)
print(f"  Sharpe Ratio   : {sharpe:.3f}")
print(f"  Max Drawdown   : {max_dd:.2%}")
print(f"  Win Rate       : {win_rate:.2%}")
print(f"  Profit Factor  : {profit_factor:.3f}")
print(f"  Total Trades   : {(positions != 0).sum()}")
print(f"  ---")
print(f"  B&H Sharpe     : {bh_sharpe:.3f}")
print(f"  Alpha (vs B&H) : {sharpe - bh_sharpe:.3f}")

# --- Equity Curve ---
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

ax1.plot(cumulative, label=f"Strategy (SR={sharpe:.2f})", color="steelblue", linewidth=1.5)
ax1.plot(bh_cumulative, label=f"Buy & Hold (SR={bh_sharpe:.2f})", color="gray", linewidth=1, alpha=0.7)
ax1.set_ylabel("Cumulative Return")
ax1.set_title(f"Equity Curve — {best_name}")
ax1.legend()
ax1.grid(alpha=0.3)

ax2.fill_between(range(len(drawdown)), drawdown, color="red", alpha=0.3)
ax2.set_ylabel("Drawdown")
ax2.set_xlabel("Bar")
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(str(Path(cfg.output_dir) / "backtest_equity.png"), dpi=150)
plt.show()

# --- Position distribution ---
fig, ax = plt.subplots(figsize=(6, 4))
pos_labels = ["SHORT", "FLAT", "LONG"]
pos_counts = [np.sum(positions == -1), np.sum(positions == 0), np.sum(positions == 1)]
ax.bar(pos_labels, pos_counts, color=["#e74c3c", "#95a5a6", "#2ecc71"])
ax.set_ylabel("Count")
ax.set_title("Position Distribution")
for i, v in enumerate(pos_counts):
    ax.text(i, v + 5, str(v), ha="center", fontweight="bold")
plt.tight_layout()
plt.savefig(str(Path(cfg.output_dir) / "position_distribution.png"), dpi=150)
plt.show()

## Cell 13 — Deployment Instructions

How to deploy the trained model to the MQL5 EA.

In [ ]:
# ============================================================
# Cell 13 — Deployment Instructions
# ============================================================

deployment_guide = """
╔══════════════════════════════════════════════════════════════════╗
║              DEPLOYMENT GUIDE — ONNX Model → MQL5 EA           ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  1. FILES TO DEPLOY                                              ║
║     ─────────────────                                            ║
║     • model.onnx  → Resources/model.onnx                        ║
║     • scaler.bin  → EAModels/ONNX/scaler.bin  (FILE_COMMON)     ║
║                                                                  ║
║  2. COLD DEPLOY (EA restart required)                            ║
║     ─────────────────────────────                                ║
║     a) Copy model.onnx to:                                       ║
║        <MT5_Data_Folder>/MQL5/Resources/model.onnx               ║
║     b) Copy scaler.bin to:                                       ║
║        <MT5_Common_Data>/EAModels/ONNX/scaler.bin                ║
║     c) Recompile the EA (the model is embedded as resource)      ║
║     d) Restart the EA                                            ║
║                                                                  ║
║  3. HOT-SWAP (no EA restart)                                     ║
║     ──────────────────────                                       ║
║     a) Copy new model to:                                        ║
║        <MT5_Common_Data>/EAModels/ONNX/model_update.onnx         ║
║     b) Copy new scaler to:                                       ║
║        <MT5_Common_Data>/EAModels/ONNX/scaler.bin                ║
║     c) The EA detects the file change automatically              ║
║     d) Shadow mode runs both models for 100 bars                 ║
║     e) If shadow accuracy > active accuracy + 1%, promote       ║
║     f) Otherwise, reject and keep the active model               ║
║                                                                  ║
║  4. VERIFICATION STEPS                                           ║
║     ───────────────────                                          ║
║     After deployment, check the EA logs for:                     ║
║     • [ONNX] Model initialized                                   ║
║     • [ONNX] Loaded scaler parameters | features=57              ║
║     • [AI-VOTE][ONNX] ... (regular vote heartbeat)              ║
║     • [ONNX] Hot-swap shadow model armed (if hot-swapped)       ║
║                                                                  ║
║  5. MODEL SPECIFICATION                                         ║
║     ──────────────────                                           ║
║     • Input shape:  (1, 60, 57)  float32                        ║
║     • Output shape: (1, 3)  float32  (raw logits)               ║
║     • Classes: 0=SELL, 1=NEUTRAL, 2=BUY                         ║
║     • ONNX opset: 12                                             ║
║     • Scaler: StandardScaler, 57 means + 57 scales              ║
║       Binary format: int32(57) + float64[57] + float64[57]      ║
║                                                                  ║
║  6. FEATURE NOTES                                               ║
║     ─────────────                                                ║
║     • Feature 55 (order_flow_imbalance): 0.0 in training,       ║
║       computed from tick data at runtime via ComputeOFI()        ║
║     • Feature 56 (spike_time_norm): 1.0 in training,            ║
║       computed from tick data at runtime for synthetic symbols   ║
║     • All features clipped to [-10, +10] in both Python and MQL5 ║
║                                                                  ║
╚══════════════════════════════════════════════════════════════════╝
"""

print(deployment_guide)

# --- Final output summary ---
output_dir = Path(cfg.output_dir)
print("\nOutput files:")
for f in sorted(output_dir.glob("*")):
    if f.is_file() and not f.name.startswith("."):
        size_kb = f.stat().st_size / 1024
        print(f"  {f.name:40s} {size_kb:8.1f} KB")

print(f"\n{'='*60}")
print(f"PIPELINE COMPLETE")
print(f"{'='*60}")
print(f"Best model     : {best_name}")
print(f"Val IC         : {results[best_name]['val_ic']:.4f}")
print(f"Val Accuracy   : {results[best_name]['accuracy']:.4f}")
print(f"CPCV PSR       : {psr_val:.3f}")
print(f"Deploy gate    : {'PASS' if deploy_gate else 'FAIL'}")
if not deploy_gate:
    print(f"\nWARNING: Deployment gate FAILED. Consider:")
    print(f"  - Increasing training data")
    print(f"  - Adjusting triple-barrier parameters (k, vertical_bars)")
    print(f"  - Trying different model architectures")
    print(f"  - Use --force-export in train_model.py to override")